# Amplitude Analysis (`A_{\phi T}`)

This notebook computes and visualizes the ISW-lensing collapsed-bispectrum amplitude proxy
`A_{\phi T}` for the thesis scenarios.

Run order:
1. Run all cells top-to-bottom.
2. Do not skip Cell 1 (it defines shared objects used by later cells).

Prerequisites:
- Upstream simulation products must already exist in your `PLENS` runtime tree.
- Environment variables / `env_config.py` must point to your runtime paths.
- Use the same Python environment as the PP/PT notebooks.

Main outputs generated by this notebook:
- `fig_AphiT_hist_noiseless.png`
- `fig_AphiT_hist_noisy.png`
- `fig_AphiT_kde_only.png`
- `fig_AphiT_nulltest_forest.png`


## Cell 1: Build `A_{\phi T}` Distributions and Histogram Figures

What this cell does:
- Loads noiseless and noisy scenarios.
- Computes unbinned `C_\ell^{\phi T}` simulation arrays.
- Builds inverse-variance weights and estimates `A_{\phi T}` per simulation.
- Prints per-scenario mean and standard deviation.
- Saves histogram figures for noiseless and noisy runs.

Outputs from this cell:
- In-memory dictionaries: `A_vals`, `stats`.
- Files: `fig_AphiT_hist_noiseless.png`, `fig_AphiT_hist_noisy.png`.


In [1]:
import os
import sys
from pathlib import Path

repo_root = Path(os.environ.get("DELENSING_REPO_ROOT", "/home3/p283342/Delensing/clean-delensing")).expanduser()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
from matplotlib.lines import Line2D
import thesis_plot_style as tps

repo_root = Path(os.environ.get("DELENSING_REPO_ROOT", "/home3/p283342/Delensing/clean-delensing")).expanduser()
cache_dir = Path(os.environ.get("AMPLITUDE_CACHE_DIR", repo_root / "THESIS" / "cache" / "amplitude_results"))

noiseless_path = cache_dir / "noiseless_amplitude_hist_agr2.npz"
noisy_path = cache_dir / "noisy_amplitude_hist_agr2.npz"
for path in (noiseless_path, noisy_path):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing cache file: {path}\n"
            "Run: sbatch compute_amplitude_plot_data.slurm"
        )

noiseless = np.load(noiseless_path, allow_pickle=True)
noisy = np.load(noisy_path, allow_pickle=True)

labels = [str(x) for x in noiseless["labels"]]
A_vals = {
    "noiseless": {labels[0]: noiseless["a0"], labels[1]: noiseless["a1"], labels[2]: noiseless["a2"]},
    "noisy": {labels[0]: noisy["a0"], labels[1]: noisy["a1"], labels[2]: noisy["a2"]},
}
stats = {
    "noiseless": {
        labels[0]: (float(noiseless["mean0"]), float(noiseless["std0"])),
        labels[1]: (float(noiseless["mean1"]), float(noiseless["std1"])),
        labels[2]: (float(noiseless["mean2"]), float(noiseless["std2"])),
    },
    "noisy": {
        labels[0]: (float(noisy["mean0"]), float(noisy["std0"])),
        labels[1]: (float(noisy["mean1"]), float(noisy["std1"])),
        labels[2]: (float(noisy["mean2"]), float(noisy["std2"])),
    },
}

for noise_key in stats:
    print(f"\n=== {noise_key.upper()} ===")
    for label, (m, s) in stats[noise_key].items():
        print(f"{label:45s} mean = {m:.3f}   sigma = {s:.3f}")

abbr = {
    "MV: lensed": "Lensed",
    "MV: internally delensed with MV-QEST": "MV-QE",
    "TT: internally delensed with Pol-QEST": "Pol-QE",
}

tps.apply_style("single")
for noise_key in ["noiseless", "noisy"]:
    fname = f"fig_AphiT_hist_{noise_key}.png"
    with tps.context_figure(filename=fname) as (fig, ax):
        for color, label in zip(tps.CB_PALETTE, labels):
            ax.hist(A_vals[noise_key][label], bins=30, density=True, histtype="step", color=color)

        ax.set(
            xlabel=r"$A_{\phi T}$",
            ylabel="Probability Density",
            title=f"Distribution of $A_{{\phi T}}$ ({noise_key})",
        )

        handles, legend_labels = [], []
        for color, label in zip(tps.CB_PALETTE, labels):
            m, s = stats[noise_key][label]
            handles.append(Line2D([0], [0], color=color, lw=1.8))
            legend_labels.append(f"{abbr.get(label, label)}: mean={m:.3f}, sigma={s:.3f}")

        ax.legend(handles, legend_labels, loc="upper right", fontsize=9, frameon=False)

print(f"Amplitude cache dir: {cache_dir}")



=== NOISELESS ===
MV: lensed                                    mean = 0.988   sigma = 0.198
MV: internally delensed with MV-QEST          mean = 0.175   sigma = 0.043
TT: internally delensed with Pol-QEST         mean = 0.423   sigma = 0.207

=== NOISY ===
MV: lensed                                    mean = 0.959   sigma = 0.299
MV: internally delensed with MV-QEST          mean = 0.485   sigma = 0.169
TT: internally delensed with Pol-QEST         mean = 0.847   sigma = 0.334
Amplitude cache dir: /home3/p283342/Delensing/clean-delensing/THESIS/cache/amplitude_results


## Cell 2: Noiseless Variance-Uncertainty and Significance Check

What this cell does:
- Splits noiseless `A_{\phi T}` samples into batches.
- Estimates uncertainty on `\sigma(A_{\phi T})` from batch-to-batch scatter.
- Reports significance of variance reduction from lensed to MV-QEST delensed.

Dependency:
- Requires `A_vals` from Cell 1.


In [2]:
import numpy as np

batch_size = 16
n_batches = 240 // batch_size

sigma_of_sigma = {}
full_sigma = {}

for label in A_vals["noiseless"]:
    arr = A_vals["noiseless"][label]
    batches = arr.reshape(n_batches, batch_size)
    batch_sigmas = batches.std(axis=1, ddof=1)
    sig_full = arr.std(ddof=1)
    sig_unc = batch_sigmas.std(ddof=1)

    full_sigma[label] = sig_full
    sigma_of_sigma[label] = sig_unc
    print(f"{label:45s} sigma = {sig_full:.3f} +/- {sig_unc:.3f}")

s1, ds1 = full_sigma["MV: lensed"], sigma_of_sigma["MV: lensed"]
s2, ds2 = full_sigma["MV: internally delensed with MV-QEST"], sigma_of_sigma["MV: internally delensed with MV-QEST"]
delta = s1 - s2
sdelta = np.sqrt(ds1**2 + ds2**2)
print(f"\nNoiseless variance drop: Delta sigma = {delta:.3f} +/- {sdelta:.3f} -> {delta/sdelta:.1f} sigma")


MV: lensed                                    sigma = 0.198 +/- 0.026
MV: internally delensed with MV-QEST          sigma = 0.043 +/- 0.009
TT: internally delensed with Pol-QEST         sigma = 0.207 +/- 0.044

Noiseless variance drop: Delta sigma = 0.155 +/- 0.028 -> 5.6 sigma


## Cell 3: Noisy Variance-Uncertainty and Significance Check

What this cell does:
- Repeats the same batch-based uncertainty check as Cell 2,
  but for noisy simulations.

Dependency:
- Requires `A_vals` from Cell 1.


In [3]:
import numpy as np

batch_size = 16
n_batches = 240 // batch_size

sigma_of_sigma = {}
full_sigma = {}

for label in A_vals["noisy"]:
    arr = A_vals["noisy"][label]
    batches = arr.reshape(n_batches, batch_size)
    batch_sigmas = batches.std(axis=1, ddof=1)
    sig_full = arr.std(ddof=1)
    sig_unc = batch_sigmas.std(ddof=1)

    full_sigma[label] = sig_full
    sigma_of_sigma[label] = sig_unc
    print(f"{label:45s} sigma = {sig_full:.3f} +/- {sig_unc:.3f}")

s1, ds1 = full_sigma["MV: lensed"], sigma_of_sigma["MV: lensed"]
s2, ds2 = full_sigma["MV: internally delensed with MV-QEST"], sigma_of_sigma["MV: internally delensed with MV-QEST"]
delta = s1 - s2
sdelta = np.sqrt(ds1**2 + ds2**2)
print(f"\nNoisy variance drop: Delta sigma = {delta:.3f} +/- {sdelta:.3f} -> {delta/sdelta:.1f} sigma")


MV: lensed                                    sigma = 0.299 +/- 0.045
MV: internally delensed with MV-QEST          sigma = 0.169 +/- 0.030
TT: internally delensed with Pol-QEST         sigma = 0.334 +/- 0.055

Noisy variance drop: Delta sigma = 0.130 +/- 0.054 -> 2.4 sigma


## Cell 4: Noisy KDE Figure and Planck-Data Reference Lines

What this cell does:
- Computes/loads noisy `A_{\phi T}` distributions for each pipeline.
- Draws KDE curves for simulation distributions.
- Overlays dashed vertical lines for corresponding Planck-data amplitude estimates.

Output file:
- `fig_AphiT_kde_only.png`


In [4]:
import os
import sys
from pathlib import Path

repo_root = Path(os.environ.get("DELENSING_REPO_ROOT", "/home3/p283342/Delensing/clean-delensing")).expanduser()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
import thesis_plot_style as tps

repo_root = Path(os.environ.get("DELENSING_REPO_ROOT", "/home3/p283342/Delensing/clean-delensing")).expanduser()
cache_dir = Path(os.environ.get("AMPLITUDE_CACHE_DIR", repo_root / "THESIS" / "cache" / "amplitude_results"))
path = cache_dir / "noisy_amplitude_kde_agr2.npz"
if not path.exists():
    raise FileNotFoundError(
        f"Missing cache file: {path}\n"
        "Run: sbatch --export=ALL,STAGE=kde_noisy compute_amplitude_plot_data.slurm"
    )

kde = np.load(path, allow_pickle=True)
labels = [str(x) for x in kde["labels"]]
A_noisy = {labels[0]: kde["a0"], labels[1]: kde["a1"], labels[2]: kde["a2"]}
A_data = {labels[0]: float(kde["adata0"]), labels[1]: float(kde["adata1"]), labels[2]: float(kde["adata2"])}

abbr = {
    "MV: lensed": "Lensed",
    "MV: internally delensed with MV-QEST": "MV-QE",
    "TT: internally delensed with Pol-QEST": "Pol-QE",
}

tps.apply_style("single")
fig, ax = plt.subplots(figsize=(5, 4))
for color, label in zip(tps.CB_PALETTE, labels):
    sns.kdeplot(A_noisy[label], ax=ax, color=color, lw=2, alpha=0.8)
    ax.axvline(A_data[label], color=color, linestyle="--", lw=2)

ax.set(
    xlabel=r"$A_{\phi T}$",
    ylabel="Density",
    title="KDE of $A_{\phi T}$ (noisy) with Planck mission data",
)

handles, legend_labels = [], []
for color, label in zip(tps.CB_PALETTE, labels):
    handles.append(Line2D([0], [0], color=color, linestyle="--", lw=2))
    legend_labels.append(rf"{abbr.get(label, label)}: $\widehat{{A}}_{{\phi T}}$ = {A_data[label]:.3f}")

ax.legend(handles, legend_labels, loc="upper right", frameon=False, fontsize=9)
tps.savefig(fig, "fig_AphiT_kde_only.png")
plt.close(fig)


## Cell 5: Null-Test Forest Plot (`\hat{A}_{\phi T}^{\mathrm{null}}`)

What this cell does:
- Builds null cross-correlations using mismatched `\phi` and `T` maps.
- Fits null amplitudes for noiseless and noisy cases.
- Produces a compact forest-style summary plot.

Output file:
- `fig_AphiT_nulltest_forest.png`


In [5]:
import os
import sys
from pathlib import Path

repo_root = Path(os.environ.get("DELENSING_REPO_ROOT", "/home3/p283342/Delensing/clean-delensing")).expanduser()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import thesis_plot_style as tps

repo_root = Path(os.environ.get("DELENSING_REPO_ROOT", "/home3/p283342/Delensing/clean-delensing")).expanduser()
cache_dir = Path(os.environ.get("AMPLITUDE_CACHE_DIR", repo_root / "THESIS" / "cache" / "amplitude_results"))
path = cache_dir / "amplitude_nulltest_agr2.npz"
if not path.exists():
    raise FileNotFoundError(
        f"Missing cache file: {path}\n"
        "Run: sbatch --export=ALL,STAGE=nulltest compute_amplitude_plot_data.slurm"
    )

null = np.load(path, allow_pickle=True)
labels = [str(x) for x in null["labels"]]

null_stats = {
    "noiseless": {
        labels[0]: (float(null["a_noiseless_0"]), float(null["sa_noiseless_0"])),
        labels[1]: (float(null["a_noiseless_1"]), float(null["sa_noiseless_1"])),
        labels[2]: (float(null["a_noiseless_2"]), float(null["sa_noiseless_2"])),
    },
    "noisy": {
        labels[0]: (float(null["a_noisy_0"]), float(null["sa_noisy_0"])),
        labels[1]: (float(null["a_noisy_1"]), float(null["sa_noisy_1"])),
        labels[2]: (float(null["a_noisy_2"]), float(null["sa_noisy_2"])),
    },
}

abbr = {
    "MV: lensed": "Lensed",
    "MV: internally delensed with MV-QEST": "MV-QE",
    "TT: internally delensed with Pol-QEST": "Pol-QE",
}
markers = {"noiseless": "o", "noisy": "s"}

tps.apply_style("single")
fig, ax = plt.subplots(figsize=(5, 3.2))

y0 = np.arange(len(labels))
dy = 0.15
for i, label in enumerate(labels):
    for noise_key, dx in (("noiseless", -dy), ("noisy", +dy)):
        A, sA = null_stats[noise_key][label]
        ax.errorbar(
            A,
            y0[i] + dx,
            xerr=sA,
            fmt=markers[noise_key],
            ms=6,
            color=tps.CB_PALETTE[i],
            capsize=4,
        )

ax.axvline(0, color="k", linestyle="--", lw=1)

handles, legend_labels = [], []
for nk in ("noiseless", "noisy"):
    handles.append(Line2D([0], [0], marker=markers[nk], color="k", linestyle="None", markersize=6))
    legend_labels.append(nk.capitalize())
for i, label in enumerate(labels):
    handles.append(Line2D([0], [0], marker="o", color=tps.CB_PALETTE[i], linestyle="None", markersize=6))
    legend_labels.append(abbr[label])

ax.set(
    yticks=y0,
    yticklabels=[abbr[lbl] for lbl in labels],
    xlabel=r"$\widehat{A}_{\phi T}^{\mathrm{null}}$",
    xlim=(-0.03, 0.03),
    title=r"Null test: cross-corr $\phi \times T_{\mathrm{other}}$",
)
ax.invert_yaxis()
ax.legend(handles, legend_labels, loc="lower right", ncol=1, frameon=False)

tps.savefig(fig, "fig_AphiT_nulltest_forest.png")
plt.close(fig)
